# **Cancellation Analysis**

## Objectives

* Answer business requirement 1:

    *The client wants to understand cancellation patterns, trends and guest booking behaviour across their two Portugese hotels in order to identify risk factors and develop more effective cancellation defence strategies*

## Inputs

* outputs/datasets/collection/HotelBookings.csv

## Outputs

* Generate conclusions about the cancellation patterns that can be used to answer business requirement 1 and inform future processes and decisions




---

## Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

## Load Data

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/datasets/collection/HotelBookings.csv")
df.head(3)

Both reservation_status and reservation_status_date are data leaks against the target variable since `reservation_status == "Canceled"` and a reservation_status_date prior to the checkout date would signal to any model that the booking was cancelled. It would also create unnecessary correlations to the target variable since all `is_cancelled == 1` bookings also have a reservation_status of "Canceled". This will be permanently removed at the cleaning stage.
* Remove reservation_status and reservation_status_date.

In [ ]:
df = df.drop(columns=["reservation_status", "reservation_status_date"])
df.head(3)

## Dataset Overview

In [ ]:
df.info()

* The dataset comprises 119390 rows across 30 features.

In [ ]:
from ydata_profiling import ProfileReport
profile_report = ProfileReport(df, minimal=True)
profile_report.to_notebook_iframe()

* The dataset overview shows 3.6% missing cells and all dtypes are int64, float64 or object.
* Several variables display unusual outliers. children and babies having max values of 10 each, required_car_parking_spaces having a max value of 8 and the max value of 5400 for adr is an extreme outlier. adr also shows 1 negative value and 1959 zero values. All these values should be referred to the client for verification, they will remain in place during EDA pending further instruction.
* Several of the variables are categorical-as-numeric. is_canceled and is_repeated_guest are binary in nature, arrival_date_year has 3 classes and both company and agent are numeric ID codes 
* children, babies, previous_cancellations, previous_bookings_not_canceled, days_in_waiting_list, booking_changes and required_car_parking_spaces all are heavily skewed with over 80% of the values zero
* lead_time and adr are skewed and may benefit from transformation if linear models are used
* arrival_date_week_number and arrival_date_day_of_month are reasonably evenly distributed as would be expected 
* country has 177 categories with gbr and prt accounting for 50% of bookings and the top 10 values accounting for nearly 85% of values. For plotting purposes, I will use the top 10 and group the remaining into other.
* Several of the categorical variables are dominated by one value. Potentially relevant examples are: deposit_type: "No Deposit", meal: "BB", market_segment: "Online TA", distribution_channel: "TA/TO", customer_type: "Transient"
* Several others have very sparse categories which may warrent grouping at a later stage such as market_segment where "direct", "corporate", "complimentary", "aviation" and "undefined" collectively account for less than 10% of the bookings.

---

## Target Variable Analysis

**Cancellation Distibution**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.countplot(data=df,
              x="is_canceled")
plt.show()

* The plot shows a moderate class imbalance, the following steps evaluate that further:

In [ ]:
target = df["is_canceled"]
cxl_df = pd.DataFrame({"Value": target.unique(),
                       "Frequency": target.value_counts().reset_index(drop=True),
                       "Percentage": target.value_counts(normalize=True)})

cxl_df.style.bar(subset=["Percentage"], color="darkgreen").format({"Percentage": "{:.0%}"})

* The overall cancellation rate is 37% indicating a moderate class imbalance that will require consideration during modelling.

**Cancellation by hotel type**

In [ ]:
sns.countplot(data=df,
              x="hotel",
              hue="is_canceled")
plt.show()

* The city hotel shows both more bookings and more cancellations that the resort hotel. The city hotel appears to have a higher cancellation rate than the resort hotel.

In [ ]:
df_by_hotel = pd.DataFrame({"cancellations": target.groupby(df["hotel"]).sum(),
                            "total_bookings": target.groupby(df["hotel"]).size()}).reset_index()
df_by_hotel["cancellation_rate"] = df_by_hotel["cancellations"] / df_by_hotel["total_bookings"]
df_by_hotel["percentage_of_total_cancellations"] = df_by_hotel["cancellations"] / df_by_hotel["cancellations"].sum()

df_by_hotel.drop(columns=["total_bookings"]).style.bar(
    subset=["cancellation_rate", "percentage_of_total_cancellations"], 
    color="darkgreen").format({"cancellation_rate": "{:.0%}","percentage_of_total_cancellations": "{:.0%}"})

* This confirms that the city hotel has a higher percentage of cancellations compared to the resort hotel. This is, in part, offset by the increase in total bookings. While the city hotel accounts for 75% of all cancellations, their cancellation rate is 42% - only slightly higer than the overall cancellation rate of 37%, but significantly higher statistically than the resort hotel at 28%

**Periodicity Study**

* Check date dtypes

In [ ]:
df[["arrival_date_year", "arrival_date_month"]].dtypes

* Month dtype is object, this needs to change to categorical to maintain calendar order.


In [ ]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

df["arrival_date_month"] = pd.Categorical(df["arrival_date_month"], categories=month_order, ordered=True)
df[["arrival_date_year", "arrival_date_month"]].dtypes  # Confirm the changes 

* Prepare monthly and yearly cancellation rates for plotting

In [ ]:
df_by_year = pd.DataFrame({"year": df["arrival_date_year"].unique(),
                           "cancellation_rate": target
                           .groupby(df["arrival_date_year"])
                           .mean()
                           .reset_index(drop=True)
                           })
df_by_year.style.format({"cancellation_rate": "{:.0%}"})

In [ ]:

df_by_month = pd.DataFrame({"cancellation_rate": target
                           .groupby([df["arrival_date_year"], df["arrival_date_month"]], observed=True)
                           .mean()
                           })
df_by_month.style.format({"cancellation_rate": "{:.0%}"})

In [ ]:
import matplotlib.ticker as mticker

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle("Cancellations Over Time")
ax1.set_title("Cancellation Rate By Year")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.set_title("Cancellation Rate By Month")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax2.axhline(df_by_month["cancellation_rate"].mean(), color="red", label="mean")

sns.barplot(data=df_by_year,
            x="year",
            y="cancellation_rate",
            ax=ax1)

sns.lineplot(data=df_by_month,
             x="arrival_date_month",
             y="cancellation_rate",
             hue="arrival_date_year",
             palette="winter",
             ax=ax2)

plt.tight_layout()
plt.show()

* The cancellation rate per year is very balanced
* The cancellation rate per month shows some patterns worth further exploration, though the incomplete nature of 2023 and 2025 make it difficult to draw any definitive conclusions.

---

## Feature Analysis

* Visualise the features individually to understand their distribution before introducing the target variable.

**Numeric Features**

In [ ]:
def numeric_features(df):
    col_list = []
    for col in df.columns.to_list():
        if df[col].dtypes == "int64" or df[col].dtypes == "float64":
            col_list.append(col)
    return col_list       
        
col_list = numeric_features(df)
print(col_list)
len(col_list)

In [ ]:
import math

def plot_numeric_features(cols):
    ncols = 4
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 15))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.histplot(data=df,
                     x=col,
                     kde=True,
                     ax=axs[i])

    plt.tight_layout()

plot_numeric_features(col_list)

* The visual plots reinforce the findings from the profile report.

**Categoric Features**

In [ ]:
def categoric_features(df):
    col_list = []
    for col in df.columns.to_list():
        if df[col].dtypes == "object" or df[col].dtypes == "category":
            col_list.append(col)
    return col_list     

col_list = categoric_features(df)
col_list.remove("country") # Remove from list to plot separately
print(col_list)
len(col_list)  


* Create dataframe of top ten countries with the remainder combined into "Other"

In [ ]:
country = df["country"].value_counts(sort=True, ascending=False)
country_top_10 = country.iloc[:10]
other = pd.Series(country.iloc[10:].sum(), index=["Other"])
country = pd.concat([country_top_10, other])

country = pd.DataFrame(country, columns=["count"])
country = country.reset_index()
country = country.rename(columns={"index": "country"})
country

In [ ]:
def plot_categoric_features(cols):
    ncols = 2
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 12))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.countplot(data=df,
                     x=col,
                     ax=axs[i])
        
    sns.barplot(data=country,
                x="country",
                y="count",
                ax=axs[-1])    

    plt.tight_layout()

plot_categoric_features(col_list)

* The city hotel accounts for approximately 2/3 of the total bookings
* arrival_date_month displays a reasonably even distribution following expected seasonal pattern
* Domestic bookings (from within Portugal) account for 40% of all arrivals

---

## Bivariate Analysis

* Plot each feature of interest against is_canceled.
    * H1 feature of interest: deposit_type
    * H2 feature of interest: lead_time
    * H3 features of interest: distribution_channel and market_segment - while market_segment specifically references OTAs, it is worth plotting distribution_channel also since TA/TO covers the same bookings while broadening the scope  

**deposit_type vs is_canceled**

In [ ]:
h1_df = pd.crosstab(df["deposit_type"], target, normalize="index")
h1_df.columns = ["not_canceled", "canceled"]

h1_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
h1_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.show()

* A surprising result on the percentage of non-refundable bookings cancelled - the expectation here was that pre-paid, non-refundable bookings would be less likely to cancel.
* The data as presented does not support H1 insofar as bookings with a refundable deposit cancel only marginally less than bookings with no deposit. Bookings with non refundable deposits are almost guaranteed to cancel.

* Due to the unexpected nature of the non-refund category and the large number of unresolved duplicates, this will be run again without the duplicate bookings.

In [ ]:
h1_unique_df = df.drop_duplicates(keep="first")
h1_unique_df = pd.crosstab(h1_unique_df["deposit_type"], target, normalize="index")
h1_unique_df.columns = ["not_canceled", "canceled"]

h1_unique_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

* While there has been a 4% shift, the overall picture remains much the same for non-refundable bookings with a cancellation rate still in excess of 90%. This suggests that potential duplicates merely inflate an existing pattern rather than create an anomaly.

**lead_time vs is_canceled**

In [ ]:
sns.violinplot(data=df, x="is_canceled", y="lead_time", hue="hotel")
plt.show()

* Cancelled bookings show a higher median lead time and a noticeably wider, flatter spread than non-cancelled bookings for both hotel types. This supports H2 - bookings made further in advance are more likely to cancel. Non-cancelled bookings are heavily concentrated near zero lead time, visible in the sharp bulge at the base of both violins.

* To further explore this, the data will be split into time buckets as follows:
    * Last Minute: 0-7 days
    * Short Range: 8-30 days
    * Mid Range: 31-90 days
    * Long Range: More than 90 days

In [ ]:
import numpy as np

bins = [-np.inf, 7 ,30 ,90, np.inf]
lead_time = pd.cut(df["lead_time"], bins, labels=["Last Minute", "Short Range", "Mid Range", "Long Range"])
lead_time

In [ ]:
h2_df = df.copy()
h2_df["lead_time"] = lead_time
h2_df = pd.crosstab(h2_df["lead_time"], target, normalize="index")
h2_df.columns = ["not_canceled", "canceled"]

h2_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
h2_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.show()

* There is a clear relationship demostrated between booking lead time and the likelihood of cancellation. The cancellation rate rises steadily across each bucket. Last minute bookings (lead time 0-7 days) have only a 10% cancellation rate whereas the long range bookings (90+ days) cancel more than they arrive at 51% cancellations. The data supports H2.

**market_segment vs is_cancelled**

In [ ]:
h3_df_a = pd.crosstab(df["market_segment"], target, normalize="index")
h3_df_a.columns = ["not_canceled", "canceled"]

h3_df_a.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
h3_df_a.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.show()

* Groups have the highest cancellation rate. Online TA is at 37% which is reasonable and the highest of the non-group bookings though not significantly different from offline TA/TO.

* Next we consider the distribution_channel since this groups online and offline TA/TO together

In [ ]:
h3_df_b = pd.crosstab(df["distribution_channel"], target, normalize="index")
h3_df_b.columns = ["not_canceled", "canceled"]

h3_df_b.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
h3_df_b.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.show()

* The combined TA/TO channel shows cancellation rate of 41%. This is higher than that of the OTA alone, but the dominance of both the TA/TO and the Online TA in their feature means that while the cancellation rates are moderate, the majority of overall cancellations will be from these categories.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12, 12))
fig.suptitle("Total Cancellations By Market Segement and Distribution Channel")
ax1.set_title("Total Cancellations by Market Segment")
ax2.set_title("Total Cancellations by Distribution Channel")

sns.countplot(data=df,
              x="market_segment",
              hue="is_canceled",
              ax=ax1)

sns.countplot(data=df,
              x="distribution_channel",
              hue="is_canceled",
              ax=ax2)

plt.tight_layout()
plt.show()

* This confirms that the largest number of cancellations, by a significant margin, are attributable to the Online TA market segment and the TA/TO distribution channel
* The data supports H3 that cancellations are more likely to come from Online TA bookings

---

## Conclusions

**Dataset Overview**
* The dataset comprises 119,390 bookings across 32 features.
* Missingness is limited at 3.6% of cells though concentrated in specific features, primarily agent and country.
* Data quality issues have been referred to the client for clarification (documented in the [rm_analysis](/jupyter_notebooks/03_rm_analysis.ipynb) notebook).<br>

**Target Variable**
* The overall cancellation rate is 37% representing a moderate class imbalance that will require consideration during modelling.
* The city hotel bookings cancel at 42% compared to 28% for the resort hotel bookings. This is a meaningful difference suggesting this feature will be significant in the model.<br>

**BR1 Answers**
* Cancellations are higher in pre-paid bookings and bookings with longer lead times. 
* Bookings from online travel agents and from all travel agents/tour operators account for the highest volume of cancellations.<br>

**Hypothesis Verdicts**
* H1 - *No-deposit bookings cancel more than pre-paid bookings*: Not supported as stated. 
    * Non-refundable bookings cancel at over 90%, which inverts the expected relationship.
    * Refundable and no-deposit bookings cancel at broadly similar rates.
    * This pattern persists when duplicate rows are excluded, suggesting the duplicates inflate the pattern, but do not cause it.
* H2 - *Longer lead times are associated with higher cancellation rates*: Supported.
    * The violin plot shows that cancelled bookings have a higher median lead time and a flatter, wider distribution than non-cancelled bookings across both hotels.
    * When bucketed, the cancellation rate rises steadily from 10% for last-minute bookings (0-7 days) to 51% for long-range bookings (90+ days)
* H3 - *OTA bookings cancel more than direct bookings*: Somewhat supported.
    * Online TA cancels at 37% and combined TA/TO at 41%, both higher than direct bookings.
    * Volumetrically, Online TA accounts for the large majority of all cancellations, this pattern is repeated with TA/To bookings.
* H4 - *Distinct guest segments with meaningfully different cancellation profiles exist*: Outside the scope of this notebook - deferred to the [cluster analysis](/jupyter_notebooks/06_cluster_analysis.ipynb) notebook.<br>

**Data Quality Issues**
* The following issues are flagged for resolution in the [data cleaning](/jupyter_notebooks/04_data_cleaning.ipynb) notebook:
    * Missing values in agent, country, children and company
    * Extreme outliers in adr (one negative value, extreme high values), children, babies and required_car_parking_spaces
    * Duplicate rows — retention/removal decision (documented in the [rm analysis](/jupyter_notebooks/03_rm_analysis.ipynb) notebook)
    * Sparse categories in country, market_segment and potentially other categoricals — consider grouping at feature engineering stage

[Correlation analysis](/jupyter_notebooks/05_correlation_analysis.ipynb) is deferred until after cleaning.
